In [ ]:
import numpy as np

import librosa

import tensorflow as tf

import json

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.preprocessing.text import tokenizer_from_json

In [ ]:
from google.colab import drive

drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
speech_model = tf.keras.models.load_model(
    "/content/gdrive/MyDrive/Emotion_Project/saved_models/speech_model_fixed.keras"
)

In [ ]:
text_model = tf.keras.models.load_model(
    "/content/gdrive/MyDrive/Emotion_Project/saved_models/gru_text_emotion_model.keras"
)

In [ ]:
with open(
    "/content/gdrive/MyDrive/Emotion_Project/saved_models/text_tokenizer.json"
) as f:

    data = json.load(f)

    tokenizer = tokenizer_from_json(data)

In [ ]:
emotion_labels = [

    "angry",

    "disgust",

    "fear",

    "happy",

    "neutral",

    "sad",

    "surprise"
]

In [ ]:
def extract_mfcc(audio_path):

    audio, sample_rate = librosa.load(
        audio_path,
        sr=22050
    )

    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=sample_rate,
        n_mfcc=40
    )

    mfcc = mfcc.T

    max_pad_length = 130

    if mfcc.shape[0] < max_pad_length:

        pad_width = max_pad_length - mfcc.shape[0]

        mfcc = np.pad(
            mfcc,
            pad_width=((0, pad_width), (0, 0)),
            mode='constant'
        )

    else:

        mfcc = mfcc[:max_pad_length, :]

    return mfcc

In [ ]:
def predict_fused_emotion(audio_path, text):

    # =========================
    # Speech prediction
    # =========================

    mfcc = extract_mfcc(audio_path)

    mfcc = np.expand_dims(
        mfcc,
        axis=0
    )

    speech_probs = speech_model.predict(mfcc)[0]

    speech_class = np.argmax(speech_probs)

    speech_emotion = emotion_labels[speech_class]

    speech_confidence = np.max(speech_probs)



    # =========================
    # Text prediction
    # =========================

    sequence = tokenizer.texts_to_sequences(
        [text]
    )

    padded = pad_sequences(
        sequence,
        maxlen=50,
        padding='post'
    )

    text_probs = text_model.predict(padded)[0]

    text_class = np.argmax(text_probs)

    text_emotion = emotion_labels[text_class]

    text_confidence = np.max(text_probs)



    # =========================
    # Weighted Fusion
    # =========================

    final_probs = (
        0.7 * speech_probs +
        0.3 * text_probs
    )

    fused_class = np.argmax(final_probs)

    fused_emotion = emotion_labels[fused_class]

    fused_confidence = np.max(final_probs)



    return (

        fused_emotion,

        fused_confidence,

        speech_emotion,

        speech_confidence,

        text_emotion,

        text_confidence
    )

In [ ]:
# Update this path to a real TESS file on your Drive
audio_path = '/content/OAF_base_angry.wav'

# Extract word from filename: 'OAF_base_angry.wav' → 'base'
word = audio_path.split('/')[-1].replace('.wav', '').split('_')[1]
print(f'Audio: {audio_path}')
print(f'Word: {word}')

(fused, fused_conf,
 speech, speech_conf,
 text, text_conf) = predict_fused_emotion(audio_path, word)

print(f'\nSpeech prediction : {speech:15s}  confidence: {speech_conf:.4f}')
print(f'Text prediction   : {text:15s}  confidence: {text_conf:.4f}')
print(f'Fused prediction  : {fused:15s}  confidence: {fused_conf:.4f}')

Audio: /content/OAF_base_angry.wav
Word: base
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step

Speech prediction : angry            confidence: 0.9999
Text prediction   : angry            confidence: 0.1459
Fused prediction  : angry            confidence: 0.7437


In [ ]:
# Update this path to a real TESS file on your Drive
audio_path = '/content/OAF_back_happy.wav'

# Extract word from filename: 'OAF_base_angry.wav' → 'base'
word = audio_path.split('/')[-1].replace('.wav', '').split('_')[1]
print(f'Audio: {audio_path}')
print(f'Word: {word}')

(fused, fused_conf,
 speech, speech_conf,
 text, text_conf) = predict_fused_emotion(audio_path, word)

print(f'\nSpeech prediction : {speech:15s}  confidence: {speech_conf:.4f}')
print(f'Text prediction   : {text:15s}  confidence: {text_conf:.4f}')
print(f'Fused prediction  : {fused:15s}  confidence: {fused_conf:.4f}')

Audio: /content/OAF_back_happy.wav
Word: back
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step

Speech prediction : happy            confidence: 1.0000
Text prediction   : happy            confidence: 0.1451
Fused prediction  : happy            confidence: 0.7435


In [ ]:
# Update this path to a real TESS file on your Drive
audio_path = '/content/OAF_chair_disgust.wav'

# Extract word from filename: 'OAF_base_angry.wav' → 'base'
word = audio_path.split('/')[-1].replace('.wav', '').split('_')[1]
print(f'Audio: {audio_path}')
print(f'Word: {word}')

(fused, fused_conf,
 speech, speech_conf,
 text, text_conf) = predict_fused_emotion(audio_path, word)

print(f'\nSpeech prediction : {speech:15s}  confidence: {speech_conf:.4f}')
print(f'Text prediction   : {text:15s}  confidence: {text_conf:.4f}')
print(f'Fused prediction  : {fused:15s}  confidence: {fused_conf:.4f}')

Audio: /content/OAF_chair_disgust.wav
Word: chair
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step

Speech prediction : disgust          confidence: 0.9999
Text prediction   : disgust          confidence: 0.1451
Fused prediction  : disgust          confidence: 0.7435


In [ ]:
# Update this path to a real TESS file on your Drive
audio_path = '/content/OAF_burn_sad.wav'

# Extract word from filename: 'OAF_base_angry.wav' → 'base'
word = audio_path.split('/')[-1].replace('.wav', '').split('_')[1]
print(f'Audio: {audio_path}')
print(f'Word: {word}')

(fused, fused_conf,
 speech, speech_conf,
 text, text_conf) = predict_fused_emotion(audio_path, word)

print(f'\nSpeech prediction : {speech:15s}  confidence: {speech_conf:.4f}')
print(f'Text prediction   : {text:15s}  confidence: {text_conf:.4f}')
print(f'Fused prediction  : {fused:15s}  confidence: {fused_conf:.4f}')

Audio: /content/OAF_burn_sad.wav
Word: burn
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step

Speech prediction : surprise         confidence: 1.0000
Text prediction   : angry            confidence: 0.1453
Fused prediction  : surprise         confidence: 0.7420


In [ ]:
# Update this path to a real TESS file on your Drive
audio_path = '/content/OAF_calm_fear.wav'

# Extract word from filename: 'OAF_base_angry.wav' → 'base'
word = audio_path.split('/')[-1].replace('.wav', '').split('_')[1]
print(f'Audio: {audio_path}')
print(f'Word: {word}')

(fused, fused_conf,
 speech, speech_conf,
 text, text_conf) = predict_fused_emotion(audio_path, word)

print(f'\nSpeech prediction : {speech:15s}  confidence: {speech_conf:.4f}')
print(f'Text prediction   : {text:15s}  confidence: {text_conf:.4f}')
print(f'Fused prediction  : {fused:15s}  confidence: {fused_conf:.4f}')

Audio: /content/OAF_calm_fear.wav
Word: calm
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step

Speech prediction : fear             confidence: 0.9994
Text prediction   : angry            confidence: 0.1454
Fused prediction  : fear             confidence: 0.7415
